### COCO-Stuff Distributions

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("..")  # go to project root
print(f"cwd: {os.getcwd()}")  # sanity check

from main import instantiate_from_config, DataModuleFromConfig
from omegaconf import OmegaConf
import numpy as np
from tqdm.notebook import tqdm
import torch
import json

cwd: /home/dude-desktop/dev/taming-transformers


In [ ]:
DATA_CONFIG = """
target: main.DataModuleFromConfig
params:
    num_workers: 2
    batch_size: 32
    train:
      target: taming.data.coco.CocoImagesAndCaptionsTrain
      params:
        size: 296
        crop_size: 256
        onehot_segmentation: true
        use_stuffthing: true
    validation:
      target: taming.data.coco.CocoImagesAndCaptionsValidation
      params:
        size: 256
        crop_size: 256
        onehot_segmentation: true
        use_stuffthing: true
"""

data_cfg = OmegaConf.create(DATA_CONFIG)

In [4]:
data: DataModuleFromConfig = instantiate_from_config(data_cfg)
data.prepare_data()
data.setup()
train_ldr = data.train_dataloader()
val_ldr = data.val_dataloader()

ImgToCaptions: 100%|██████████| 25014/25014 [00:00<00:00, 1386388.29it/s]


In [5]:
with open("data/cocostuffthings/labels.txt", "r") as f:
    labels = [line.strip().split(": ")[1] for line in f.readlines()]
print(labels)

['unlabeled', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'street sign', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'hat', 'backpack', 'umbrella', 'shoe', 'eye glasses', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'plate', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'mirror', 'dining table', 'window', 'desk', 'toilet', 'door', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'blender', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush', 'hair brush', 'banner', 

#### Pixel-level class distributions
For all images, display class distribution over all pixels

In [10]:
def calculate_pixel_level_distribution(loader, num_classes):
    class_counts = torch.zeros(num_classes, dtype=torch.int64, device='cpu')
    
    for batch in tqdm(loader, desc="Calculating pixel distribution"):
        try:
            segmentations = batch["segmentation"]
            index_mask = torch.argmax(segmentations, dim=3)
            counts = torch.bincount(
                index_mask.cpu().flatten(), 
                minlength=num_classes
            )
            class_counts += counts
        except Exception as e:
            print("counts shape:", counts.shape)
            print("class_counts shape:", class_counts.shape)
            raise e

    total_pixels = class_counts.sum().item()
    return class_counts.numpy(), total_pixels

num_classes = len(labels)
train_class_counts, train_total_pixels = calculate_pixel_level_distribution(train_ldr, num_classes)
val_class_counts, val_total_pixels = calculate_pixel_level_distribution(val_ldr, num_classes)

Calculating pixel distribution:   0%|          | 0/3697 [00:00<?, ?it/s]

ERROR: Unexpected bus error encountered in worker. This might be caused by insufficient shared memory (shm).
 

KeyboardInterrupt: 

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/dude-desktop/anaconda3/envs/taming/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/dude-desktop/anaconda3/envs/taming/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/dude-desktop/anaconda3/envs/taming/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_loop.start()
  File "/home/dude-desktop/anaconda3/envs/taming/lib/python3.12/site-packages/tornado/platform/asyncio.py", line 211, in start
    self.asyncio_loop.run_forever()
  File "/home/dude-desktop/anaconda3/envs/taming/lib/python3.12/asyncio/base_events.py", line 645, in run_forever
    self._run_once()
  File "/home/dude-desktop/anaconda3/envs/taming/lib/python3.12/asyncio/base_events.py", line 1961, in _

In [ ]:
data = {
    "labels": labels,
    "train": {
        "class_counts": train_class_counts.tolist(),
        "total_pixels": train_total_pixels,
    },
    "validation": {
        "class_counts": val_class_counts.tolist(),
        "total_pixels": val_total_pixels,
    }
}
with open("analysis/pixel_level_distribution.json", "w") as f:
    json.dump(data, f, indent=4)

#### Image-level class distributions
For all images, if an image has at least 1 pixel with the class, increment the counter for that class

In [ ]:
for item in train_ldr.dataset:
    print(item.keys())
    break